# Typing Info

In [1]:
from itertools import combinations, permutations
from helpers import count_of
from poketypings import *

In [2]:
from itertools import zip_longest
from random import randint

def teamify(type_list):

  if len(type_list) <= 4:
    return [Typing(type) for type in type_list]
  
  pairs = zip_longest(type_list[:4], type_list[4:])
  
  return [Typing(*[_ for _ in pair if _ is not None]) for pair in pairs]

(teamify([]),
 teamify([ FAIRY ]),
 teamify([FAIRY, WATER, ROCK, FIRE]),
 teamify([FAIRY, WATER, ROCK, FIRE, ELECTRIC]),
 teamify([DRAGON, DARK, ROCK]))

([],
 [(fairy)],
 [(fairy), (water), (rock), (fire)],
 [(fairy, electric), (water), (rock), (fire)],
 [(dragon), (dark), (rock)])

# Singles

In [3]:
('amount of teams that can be made with 3 pokemon:',
count_of(iter(combinations(ALL_TYPES, 3*2))))

('amount of teams that can be made with 3 pokemon:', 18564)

In [4]:
def count_advantages(*poketypings):
  advantage_set = set()
  for typing in poketypings:
    for subtyping in typing:
      for advantage in advantages(subtyping):
        advantage_set.add(advantage)
  return len(advantage_set)

In [5]:
from collections import defaultdict

singles_typings = iter(combinations(ALL_TYPES, 3*2))
advantage_counts = defaultdict(lambda: 0)

for team_typings in singles_typings:
  poke1 = Typing(*team_typings[:2])
  poke2 = Typing(*team_typings[2:4])
  poke3 = Typing(*team_typings[4:])

  advantage_counts[count_advantages(poke1, poke2, poke3)] += 1

{k:advantage_counts[k] for k in sorted(advantage_counts.keys())}

{6: 3,
 7: 42,
 8: 189,
 9: 719,
 10: 1827,
 11: 3261,
 12: 4513,
 13: 4176,
 14: 2705,
 15: 936,
 16: 183,
 17: 10}

💡 there is no singles team of 3 pokemon that can cover all types

# Doubles

In [6]:
('amount of teams that can be made with 4 pokemon:',
count_of(iter(combinations(ALL_TYPES, 4*2))))

('amount of teams that can be made with 4 pokemon:', 43758)

In [7]:
from collections import defaultdict

doubles_typings = iter(combinations(ALL_TYPES, 3*4))
advantage_counts = defaultdict(lambda: 0)

for team_typings in doubles_typings:
  advantage_counts[count_advantages(*teamify(team_typings))] += 1

{k:advantage_counts[k] for k in sorted(advantage_counts.keys())}

{13: 17, 14: 311, 15: 1737, 16: 4919, 17: 7490, 18: 4090}

# Creating a Team
The first version of this algorithm will just generate a team randomly; later versions will take into account synergy, weaknesses, etc

technically, i believe of my team picking formula, what were doing here is minimizing walling. let me restate what's going on:
- high coverage, means anti-walling
- high defence, means anti-sweeping
- synergy is how well pokemon work with each other, and is less about types than it is about moves abilities and stats. if i make a web-page, then i can delegate that to users.


steps for basic algo
- maintain a inverse index of all dual types and which pokemon fall into there (which is not that many - 18C2 = 153, and that assumes every combo exists; +18 for single types)
- filter through each type combo that has maximal advantage, and select a random one (theyre all equal offensively)
- then, using the inverse mapping, just select a random pokemon from each dual typing

In [8]:
from os import read
from fetch_typings import get_pokemon_typings

all_pokemon = get_pokemon_typings()

all_pokemon[:3], '...'

([{'name': 'bulbasaur', 'typing': ['grass', 'poison']},
  {'name': 'ivysaur', 'typing': ['grass', 'poison']},
  {'name': 'venusaur', 'typing': ['grass', 'poison']}],
 '...')

in terms of offensive type combos, there is no more insight to gain. hoping to build a public webpage that lets people randomly generate teams, and actually take advantage of all the kinds of teams that maximize coverage.

# Random Team Picker


## Step 1
> maintain a inverse index of all dual types and which pokemon fall into there (which is not that many - 18C2 = 153, and that assumes every combo exists; +18 for single types)


In [9]:
index = {
  Typing(*pair): []
  for pair
  in list(combinations(ALL_TYPES, 2)) + [(t,) for t in ALL_TYPES]
}

# expecting 18C2 * 2 (153 * 2)
(
  index[Typing(NORMAL, FIRE)] is index[Typing(FIRE,NORMAL)],
  len(index),
  index
)

(True,
 171,
 {(normal, fire): [],
  (normal, water): [],
  (normal, electric): [],
  (normal, grass): [],
  (normal, ice): [],
  (normal, fighting): [],
  (normal, poison): [],
  (normal, ground): [],
  (normal, flying): [],
  (normal, psychic): [],
  (normal, bug): [],
  (normal, rock): [],
  (normal, ghost): [],
  (normal, dragon): [],
  (normal, dark): [],
  (normal, steel): [],
  (normal, fairy): [],
  (fire, water): [],
  (fire, electric): [],
  (fire, grass): [],
  (fire, ice): [],
  (fire, fighting): [],
  (fire, poison): [],
  (fire, ground): [],
  (fire, flying): [],
  (fire, psychic): [],
  (fire, bug): [],
  (fire, rock): [],
  (fire, ghost): [],
  (fire, dragon): [],
  (fire, dark): [],
  (fire, steel): [],
  (fire, fairy): [],
  (water, electric): [],
  (water, grass): [],
  (water, ice): [],
  (water, fighting): [],
  (water, poison): [],
  (water, ground): [],
  (water, flying): [],
  (water, psychic): [],
  (water, bug): [],
  (water, rock): [],
  (water, ghost): [],
 

In [10]:
for pokemon in all_pokemon:
  typing = Typing(*list(map(type_name_to_type.__getitem__, pokemon['typing'])))
  index[typing].append(pokemon['name'])

(index[Typing(DRAGON)][:3],'...',
 index[Typing(FIRE, GRASS)][:3],'...')

(['dratini', 'dragonair', 'bagon'],
 '...',
 ['scovillain', 'ogerpon-hearthflame-mask', 'scovillain-mega'],
 '...')

❗️ we *need* to make sure there are no types in the index with no pokemon in them

In [11]:
if ('missing_types' not in globals()):
  print('ran')

  before = set(index.keys())

  for typing,pokemon in list(index.items()):
    if len(pokemon) == 0:
      del index[typing]

  after = set(index.keys())

  missing_types = before - after

f'{len(missing_types)} missing type combos from index:', globals()['missing_types']

ran


('9 missing type combos from index:',
 {(bug, dragon),
  (fire, fairy),
  (ground, fairy),
  (ice, poison),
  (normal, bug),
  (normal, ice),
  (normal, rock),
  (normal, steel),
  (rock, ghost)})

## Step 2
> filter through each type combo that has maximal advantage, and select a random one (theyre all equal offensively)

In [12]:
hcts = []

for team_typings in iter(combinations(ALL_TYPES, 3*4)):
  poke1 = Typing(*team_typings[:2])
  poke2 = Typing(*team_typings[2:4])
  poke3 = Typing(*team_typings[4:6])
  poke4 = Typing(*team_typings[6:8])

  if (poke1 in missing_types or poke2 in missing_types 
    or poke3 in missing_types or poke4 in missing_types):
    continue

  if count_advantages(poke1, poke2, poke3, poke4) == 18:
    hcts.append((poke1, poke2, poke3, poke4))

'example of a Highest Coverage Typing Set (HCTS):', hcts[0]

('example of a Highest Coverage Typing Set (HCTS):',
 ((normal, electric), (ice, fighting), (poison, ground), (flying, ghost)))

In [13]:
f'actual amount of HCTS: {len(hcts)}'

'actual amount of HCTS: 13'

# Step 3
> then, using the inverse mapping, just select a random pokemon from each dual typing

In [14]:
import random

random_ts = random.choice(hcts)
random_ts

((fire, grass), (ice, fighting), (poison, ground), (psychic, ghost))

In [15]:
team = []

for typing in random_ts:
  random_pokemon_with_typing = random.choice(index[typing])
  team.append(random_pokemon_with_typing)

team

['scovillain', 'crabominable-mega', 'wooper-paldea', 'lunala']

# Major!
i just realized that the current code isnt too useful. it assumes that defeinsively every pokemon has one type. but it's a good starting point for an idea of how the code works.

# Extending to Turn Into a Team Calculator

```py
def count_advantages(*poketypings):
  advantage_set = set()
  for typing in poketypings:
    for subtyping in typing:
      for advantage in advantages(subtyping):
        advantage_set.add(advantage)
  return len(advantage_set)
```

above is the code that we use to calculate hcts assuming opponents are all single typed.

--- 
# Calculator
instead, the new brute force algorithm will be
1. again, loop through all teams of 4
1. simulation - filter the teams that totally cover the opponent's team


the simulate function is the meat and potatoes:
1. the input is a team of 4 dual OR single types (shouldnt really matter)
1. the output is between 4 and 8 types that totally counter the opposing types

sub-function: disadvantages; similar to the advantages function that derives its output from the type chart
1. ...TODO

simulation
1. for each opponent pokemon, collect disadvantages into one set
1. this gives us a set of types we should use in our team
1. given this new type set (which is a subset of ALL_TYPES), generate all typing sets of size 4 {(), (), (), ()}
1. use the reverse index to randomly fill that typing set with pokemon

think through edge cases:
- what happens when their team is full of sableyes? (1 weakness); should still work since were collecting disadvantages into a set; the real edge case is what happens when typing set is less than 4 pokemon (we return and consumer notifies user)

In [16]:
INPUT_TEAM = [
  Typing(POISON, FIGHTING), #sneasler
  Typing(ROCK, FLYING),     #aerodactyl
  Typing(GROUND, DRAGON),   #garchomp
  Typing(WATER, ELECTRIC),  #rotom-wash
]

In [17]:
import inspect

print(inspect.getsource(disadvantages))

def disadvantages(*defender):
    
    if len(defender) < 2:
      defender = [defender[0], None]

    type1, type2 = defender

    N = len(type_chart)

    type1_weaknesses = [type_chart[i][type1] for i in range(N)]
    type2_weaknesses = (
        [1] * N
        if type2 is None
        else [type_chart[i][type2] for i in range(N)]
    )
    combined_weaknesses = [1] * N
    
    for attacker in range(N):
      combined_weaknesses[attacker] = (
          type1_weaknesses[attacker]
          * type2_weaknesses[attacker]
      )

    super_effective_filter = filter(
        lambda i: combined_weaknesses[i] > 1,
        range(N)
    )
    
    return list(map(T, super_effective_filter))



calculating the disadvantages of a whole opposing team
> 1. for each opponent pokemon, collect disadvantages into one set
> 1. this gives us a set of types we should cover
> 1. given this new type set (which is a subset of ALL_TYPES), generate all typing sets of size 4 {(), (), (), ()}
> 1. use the reverse index to randomly fill that typing set with pokemon

👉 note that we wanna know the advantages against each opponent pokemon, because we eventually wanna see if any one of our pokemon can oppose 2+ of theirs

In [18]:
def team_disadvantages(team):
  weakness_map = []
  for pokemon in team:
    weakness_map.append((
      pokemon,
      disadvantages(*pokemon.typing)
    ))
  return weakness_map
  

In [19]:
opp_disadvantages = team_disadvantages(INPUT_TEAM)
opp_disadvantages

[((poison, fighting),
  [<Type.GROUND: 8>, <Type.FLYING: 9>, <Type.PSYCHIC: 10>]),
 ((rock, flying),
  [<Type.WATER: 2>,
   <Type.ELECTRIC: 3>,
   <Type.ICE: 5>,
   <Type.ROCK: 12>,
   <Type.STEEL: 16>]),
 ((ground, dragon), [<Type.ICE: 5>, <Type.DRAGON: 14>, <Type.FAIRY: 17>]),
 ((water, electric), [<Type.GRASS: 4>, <Type.GROUND: 8>])]

# TODO
calculating the disadvantages of a whole opposing team
> 1. for each opponent pokemon, collect disadvantages into one set
> 1. this gives us a set of types we should cover

we are now HERE❗️
> 1. given this new type set (which is a subset of ALL_TYPES), generate all typing sets of size 4 {(), (), (), ()}
> 1. use the reverse index to randomly fill that typing set with pokemon


❌ im not sure if this algorithm is correct. it may almost be, but here is an edge case:
- 3 normal types, 1 steel

🔑 the real algorithm:
- dont collect all weaknesses into one set
- instead map them to each opponent pokemon
- then collect them into a new mapping between weakness, and how many pokemon in the opponents team are weak
- we then pick types with the highest rank when sorted by amount of advantages against the whole team
- best case: all your pokemon answer all your opponents (imagine opponent is a rock team)
- worst case: ...need to confirm that the worst case is still 1 answer for 1 of your opponents pokemon


# Extra
- once we have our advantage list, the same way we rank which advantages are better than others, we can also rank which types have the most resistances in the opposing team
- the remaining 2 pokemon will be up to the player, ideally they will pick in a manner that 

# TL;DR
- we want the input to be a team of 4
- and we want the output to be a team of 4, where we pick each of our pokemon to maximize the effectiveness, settling for 1:1 worst case
- 👉 we want this tool to work for the entire future of pokemon: if a new type is added, if new pokemon are added, etc

In [20]:
from collections import Counter

adv_counter = Counter()

#we can assume each type only appears once per opp
for opp, weaknesses in opp_disadvantages:
  for t in weaknesses:
    adv_counter[t] += 1

sorted(adv_counter.items(), key=lambda item: item[1],reverse=True)

[(<Type.GROUND: 8>, 2),
 (<Type.ICE: 5>, 2),
 (<Type.FLYING: 9>, 1),
 (<Type.PSYCHIC: 10>, 1),
 (<Type.WATER: 2>, 1),
 (<Type.ELECTRIC: 3>, 1),
 (<Type.ROCK: 12>, 1),
 (<Type.STEEL: 16>, 1),
 (<Type.DRAGON: 14>, 1),
 (<Type.FAIRY: 17>, 1),
 (<Type.GRASS: 4>, 1)]

# Problem: How do we pick typings now?

## first approach
try everything and get the one with the least resistances

In [21]:
effective_types = list(adv_counter.keys())

print(count_of(combinations(effective_types,8)))

def get_weaknesses(types):
  poke1 = Typing(*types[:2])
  poke2 = Typing(*types[2:4])
  poke3 = Typing(*types[4:6])
  poke4 = Typing(*types[6:8])
  return ((poke1, poke2, poke3, poke4), team_disadvantages([poke1, poke2, poke3, poke4]))

def count_weaknesses(disadvantages):
  counter = 0
  for _, weaknesses in disadvantages:
    counter += len(weaknesses)
  return counter

team, weaknesses = get_weaknesses(next(iter(combinations(effective_types,8))))
min_weaknesses = count_weaknesses(weaknesses)

for team_types in combinations(effective_types,8):
  team, weaknesses = get_weaknesses(team_types)
  if count_weaknesses(weaknesses) < min_weaknesses:
    min_weaknesses = count_weaknesses(weaknesses)
    min_weakness_team = team

min_weaknesses, min_weakness_team

165


(11, ((ground, flying), (water, electric), (ice, steel), (dragon, fairy)))

## Second Approach (min weaknesses specifically to opponent)

the above code picks on minimum weaknesses in general, not minimum weaknesses relative to the team in question (that may actually be a good thing).

below ill attempt to calculate the min weaknesses to the opp team specifically.


approach
- try all teams
- generate all dis
- dont just count dis, but keep track of the dis that contained the least amount of pokemon in the opponent team

In [22]:
def count_opp_types(disadvantages, opp_types):
  counter = 0
  for _, weaknesses in disadvantages:
    for weakness in weaknesses:
      if weakness in opp_types:
        counter += 1
  return counter

opp_types = []
for opp in INPUT_TEAM:
  opp_types.extend(opp)

team, weaknesses = get_weaknesses(next(iter(combinations(effective_types,8))))
min_weaknesses = count_opp_types(weaknesses, opp_types)

for team_types in combinations(effective_types,8):
  team, weaknesses = get_weaknesses(team_types)
  if count_opp_types(weaknesses, opp_types) < min_weaknesses:
    print('updated')
    min_weaknesses = count_opp_types(weaknesses, opp_types)
    min_weakness_team = team

min_weaknesses, min_weakness_team

updated
updated
updated


(5, ((ground, flying), (psychic, water), (electric, steel), (dragon, fairy)))

In [23]:
'do an eye test, is this really 5x effective against the output?', INPUT_TEAM

('do an eye test, is this really 5x effective against the output?',
 [(poison, fighting), (rock, flying), (ground, dragon), (water, electric)])

### example

(5, ((ground, flying), (psychic, water), (electric, steel), (dragon, fairy)))


[(poison, fighting), (rock, flying), (ground, dragon), (water, electric)])


poison -> (dragon,fairy)
fighting -> (electric,steel)
ground -> (electric, steel)
water -> (ground, flying)
electric -> (psychic, water)

✅

# Core Conflict
Once we get a set of types that hurt the opponent, there are a few ways to branch.

at this point in the video, start discussing arrows from left to right (maximum advantage) and arrows right to left (maximum disadvantage)

## Maximum Advantage
❗️ what we havent done yet: picked the maximally effective team, and we need to do this.
- the algorithm is to try everything and count the advantages. DO THIS!
- practically speaking we want to maximize the number of arrows from left to right
- ...can i think of this as a graph problem where i want to maximize paths

## Minimum Disadvantage
If we want to care about resistances instead (balance; maybe make this choice a parameter that lets you pick between attack or balance)
1. Greedily pick a team of the most harmful types (easy - 4 {Ground,Flying} types for eg)
1. Break the first 8 types into 4 pokemon (downside - all eggs in 1 basket)
1. Somewhere in between
    - you can repeat types like in the greedy way, but your secondary type needs to be picked to cover your primary type's weaknesses
    - pick the top 4 primary types, then pick secondary types based on maximizing your team's resistance (im liking this more and more)
    - try all teams and select the one that is the least weak to your opponent (favorite so far because it makes a lot of sense in english, and it's not hard)


In [24]:
def count_advs(my_team, opp_team):
  count = 0
  for friend in my_team:
    for type in friend:
      for opp in opp_team:
        scalar = type_chart[type][opp.typing[0]]
        scalar *= type_chart[type][opp.typing[1]]
        if scalar > 1:
          count += 1
  return count


# note: i added [::-1] because coincidentally, in the original list, the best team was the first team, so i wanted to confirm it works
max_adv_team = teamify(next(iter(combinations(effective_types[::-1], 8))))
max_advs = count_advs(team, INPUT_TEAM)

for team_types in combinations(effective_types[::-1], 8):
  team = teamify(team_types)
  advs = count_advs(team, INPUT_TEAM)
  if advs >= max_advs:
    max_adv_team = team
    max_advs = advs

max_advs, max_adv_team

(10, [(steel, water), (rock, psychic), (ice, flying), (electric, ground)])

# Randomize the Strongest Teams!
there are 165 ways to choose 8 of 11, so there were a lot of teams that actually have 10 advantages! i didnt expect this. so now that we have max_advs, lets randomize the one that's chosen.

In [25]:
strongest_teams = list(filter(
  lambda team: count_advs(team, INPUT_TEAM) == max_advs,
  map(teamify, combinations(effective_types, 8))
))

len(strongest_teams), strongest_teams[:10]


(84,
 [[(ground, electric), (flying, ice), (psychic, rock), (water, steel)],
  [(ground, electric), (flying, ice), (psychic, rock), (water, dragon)],
  [(ground, electric), (flying, ice), (psychic, rock), (water, fairy)],
  [(ground, electric), (flying, ice), (psychic, rock), (water, grass)],
  [(ground, electric), (flying, ice), (psychic, steel), (water, dragon)],
  [(ground, electric), (flying, ice), (psychic, steel), (water, fairy)],
  [(ground, electric), (flying, ice), (psychic, steel), (water, grass)],
  [(ground, electric), (flying, ice), (psychic, dragon), (water, fairy)],
  [(ground, electric), (flying, ice), (psychic, dragon), (water, grass)],
  [(ground, electric), (flying, ice), (psychic, fairy), (water, grass)]])

In [26]:
team = random.choice(strongest_teams)
team

[(ground, rock), (psychic, steel), (water, dragon), (ice, fairy)]

In [27]:
exact_team = []

for typing in team:
  exact_team.append(random.choice(index[Typing(*typing)]))

exact_team

['larvitar', 'beldum', 'tatsugiri-curly-mega', 'ninetales-alola']

❗️❗️❗️i just had a major realization... for each combination, eg
water, rock, fire, ground

i need to consider multiple teams
```
water->rock
     ->fire
     ->ground
```
etc.

how do i do this? more combination function? i need to replace the teamify function

... i believe i need to do some backtracking. produce every team starting with water, pick every team starting with one of remaining types, etc, backtrack, etc

maybe:
```
remaining = 8
for 8C2
  remove from remaining
  for 6C2
    remove from remaining
    for 4C2
      remove from remaining
      add back to set (next iteration will choose others)
    add back to set
  add back to set
```

In [28]:
#when calling gen_subteams on S-{x,y}, generate every team that doesnt contain {x,y}
#whats left is to add them back, select another x,y, until every pair nC2 is chosen

#note that nC2 is the same as a "for i; for j=i+1" loop

def _gen_variations_ordered(sequence):
  S = set(sequence)
  return __gen_variations_ordered(S)

def __gen_variations_ordered(S):

  if len(S) == 0:
    return [[]]
  if len(S) == 1:
    return [[Typing(next(iter(S)))]]

  teams = []

  for l,r in combinations(S,2):
    S.remove(l)
    S.remove(r)
    for subteam in _gen_variations_ordered(S):
      subteam.append(Typing(l, r))
      teams.append(list(subteam))
    S.add(l)
    S.add(r)

  return teams

(
  len(_gen_variations_ordered(set([ROCK,WATER,FIRE,GRASS,DRAGON,FAIRY,ICE,FIGHTING]))),
  _gen_variations_ordered(set([ROCK,WATER,FIRE,GRASS])),
  _gen_variations_ordered(set([DARK,ROCK])),
  _gen_variations_ordered(set([DARK,ROCK,DRAGON])),
  _gen_variations_ordered(set([DARK])),
)

(2520,
 [[(rock, grass), (fire, water)],
  [(water, grass), (fire, rock)],
  [(water, rock), (fire, grass)],
  [(fire, grass), (water, rock)],
  [(fire, rock), (water, grass)],
  [(fire, water), (rock, grass)]],
 [[(rock, dark)]],
 [[(dark), (rock, dragon)],
  [(dragon), (rock, dark)],
  [(rock), (dragon, dark)]],
 [[(dark)]])

**optimizing the above function**

once we generate a subteam, we learn some info that my intuition is screaming could be useful.

In [29]:
_gen_variations_ordered(set([ROCK,WATER,FIRE,GRASS,FLYING,DRAGON]))[:5],'...'

([[(rock, dragon), (flying, grass), (fire, water)],
  [(grass, dragon), (flying, rock), (fire, water)],
  [(grass, rock), (flying, dragon), (fire, water)],
  [(flying, dragon), (grass, rock), (fire, water)],
  [(flying, rock), (grass, dragon), (fire, water)]],
 '...')

actually, to continue off of that, pretty sure we can do this through bottom up memoization. reuse answers already available.

according to my analysis of algorithms notes, we can bottom-upify if
- parameters to subproblem can be mapped to solutions (not sure if this is true yet)
- subproblem answers available before current problem (this is true)

maybe this way we can more easily see how to avoid extra iterations

given a set of typings, the result is always the same, so can map the typing set to the result

In [30]:
# def gen_variations(S):

#   if len(S) == 0:
#     return [[]]
#   if len(S) == 1:
#     return [[Typing(next(iter(S)))]]

#   teams = []

#   for l,r in combinations(S,2):
#     S.remove(l)
#     S.remove(r)
#     for subteam in gen_variations(S):
#       subteam.append(Typing(l, r))
#       teams.append(list(subteam))
#     S.add(l)
#     S.add(r)

#   return teams

'''

start
📌ROCK,WATER,FIRE,GRASS,FLYING,DRAGON

---
,WATER,FIRE,GRASS,FLYING,DRAGON,ROCK
,FIRE,GRASS,FLYING,DRAGON,ROCK,WATER
,GRASS,FLYING,DRAGON,ROCK,WATER,FIRE
,FLYING,DRAGON,ROCK,WATER,FIRE,GRASS
,DRAGON,ROCK,WATER,FIRE,GRASS,FLYING
,ROCK,WATER,FIRE,GRASS,FLYING,DRAGON
---
,ROCK,FIRE,GRASS,FLYING,DRAGON,WATER
,ROCK,GRASS,FLYING,DRAGON,WATER,FIRE
,ROCK,FLYING,DRAGON,WATER,FIRE,GRASS
,ROCK,DRAGON,WATER,FIRE,GRASS,FLYING
,ROCK,WATER,FIRE,GRASS,FLYING,DRAGON
----
,ROCK,WATER,GRASS,FLYING,DRAGON,FIRE
,ROCK,WATER,FLYING,DRAGON,FIRE,GRASS
,ROCK,WATER,DRAGON,FIRE,GRASS,FLYING
,ROCK,WATER,FIRE,GRASS,FLYING,DRAGON
---
,ROCK,WATER,FIRE,FLYING,DRAGON,GRASS
,ROCK,WATER,FIRE,DRAGON,GRASS,FLYING
,ROCK,WATER,FIRE,GRASS,FLYING,DRAGON
---
,ROCK,WATER,FIRE,GRASS,DRAGON,FLYING
,ROCK,WATER,FIRE,GRASS,FLYING,DRAGON
---
,ROCK,WATER,FIRE,GRASS,FLYING,DRAGON

why is this algo producing the same value as the input every time?

'''

'\n\nstart\n📌ROCK,WATER,FIRE,GRASS,FLYING,DRAGON\n\n---\n,WATER,FIRE,GRASS,FLYING,DRAGON,ROCK\n,FIRE,GRASS,FLYING,DRAGON,ROCK,WATER\n,GRASS,FLYING,DRAGON,ROCK,WATER,FIRE\n,FLYING,DRAGON,ROCK,WATER,FIRE,GRASS\n,DRAGON,ROCK,WATER,FIRE,GRASS,FLYING\n,ROCK,WATER,FIRE,GRASS,FLYING,DRAGON\n---\n,ROCK,FIRE,GRASS,FLYING,DRAGON,WATER\n,ROCK,GRASS,FLYING,DRAGON,WATER,FIRE\n,ROCK,FLYING,DRAGON,WATER,FIRE,GRASS\n,ROCK,DRAGON,WATER,FIRE,GRASS,FLYING\n,ROCK,WATER,FIRE,GRASS,FLYING,DRAGON\n----\n,ROCK,WATER,GRASS,FLYING,DRAGON,FIRE\n,ROCK,WATER,FLYING,DRAGON,FIRE,GRASS\n,ROCK,WATER,DRAGON,FIRE,GRASS,FLYING\n,ROCK,WATER,FIRE,GRASS,FLYING,DRAGON\n---\n,ROCK,WATER,FIRE,FLYING,DRAGON,GRASS\n,ROCK,WATER,FIRE,DRAGON,GRASS,FLYING\n,ROCK,WATER,FIRE,GRASS,FLYING,DRAGON\n---\n,ROCK,WATER,FIRE,GRASS,DRAGON,FLYING\n,ROCK,WATER,FIRE,GRASS,FLYING,DRAGON\n---\n,ROCK,WATER,FIRE,GRASS,FLYING,DRAGON\n\nwhy is this algo producing the same value as the input every time?\n\n'

...this new algo generates unordered pairs... which we actually need. after a quick check with claude to confirm how many teams of pairs i can create given the set (ROCK,DRAGON,FIRE,GRASS,WATER,FLYING), it explained to me that there are 90 ordered ones, and 15 unordered ones. so my first algo was correct (i should cross check online too - perhaps claude just used my function to run that calculation).

tl;dr - i cant optimize further, UNLESS in the index, i treat unordered pairs the same... which i think i do

In [31]:
from collections import deque

def cycle(sequence):
  og_seq = type(sequence)
  stop_at = len(sequence)
  q = deque(sequence)
  while stop_at > 0:
    yield og_seq(q)
    q.append(q.popleft())
    stop_at -= 1

In [32]:
for q in cycle([1,2,3,4]):
  print(q)

[1, 2, 3, 4]
[2, 3, 4, 1]
[3, 4, 1, 2]
[4, 1, 2, 3]


In [33]:
for q in cycle([ROCK,WATER,FIRE,GRASS,FLYING,DRAGON]):
  print(q)

[<Type.ROCK: 12>, <Type.WATER: 2>, <Type.FIRE: 1>, <Type.GRASS: 4>, <Type.FLYING: 9>, <Type.DRAGON: 14>]
[<Type.WATER: 2>, <Type.FIRE: 1>, <Type.GRASS: 4>, <Type.FLYING: 9>, <Type.DRAGON: 14>, <Type.ROCK: 12>]
[<Type.FIRE: 1>, <Type.GRASS: 4>, <Type.FLYING: 9>, <Type.DRAGON: 14>, <Type.ROCK: 12>, <Type.WATER: 2>]
[<Type.GRASS: 4>, <Type.FLYING: 9>, <Type.DRAGON: 14>, <Type.ROCK: 12>, <Type.WATER: 2>, <Type.FIRE: 1>]
[<Type.FLYING: 9>, <Type.DRAGON: 14>, <Type.ROCK: 12>, <Type.WATER: 2>, <Type.FIRE: 1>, <Type.GRASS: 4>]
[<Type.DRAGON: 14>, <Type.ROCK: 12>, <Type.WATER: 2>, <Type.FIRE: 1>, <Type.GRASS: 4>, <Type.FLYING: 9>]


In [34]:
# broken version
def create_team(type_list):
  type_list = list(type_list) #copy to avoid mutating original
  team = []
  poke = []
  while type_list:
    if len(poke) > 1:
      team.append(Typing(*poke[::-1]))
      poke.clear()
    poke.append(type_list.pop())
  if poke:
    team.append(Typing(*poke[::-1]))
  return team[::-1]


def _gen_variations_unordered(S):
  # my version - not sure why its not working yet.

  all_variations = []

  for pin in range(1,len(S)):
    prefix = S[:pin]
    remaining = S[pin:]
    for subvariant in cycle(remaining):
      variant = prefix + subvariant
      all_variations.append(create_team(variant))
    
  return all_variations

In [35]:
def _gen_variations_unordered_claude_helper(S):
    if len(S) == 0: return [[]]
    first = S[0]
    rest  = S[1:]
    result = []
    for i, partner in enumerate(rest):
        remaining = rest[:i] + rest[i+1:]
        for sub in _gen_variations_unordered_claude_helper(remaining):
            result.append([(first, partner)] + [pair for pair in sub])
    return result

def _gen_variations_unordered_claude(S):
  odd = len(S) % 2 == 1
  if odd:
    S = list(S) + [None]
  vars = _gen_variations_unordered_claude_helper(S)
  for var in vars:
    for i in range(len(var)):
      pair = var[i]
      if None in pair:
        var[i] = tuple(filter(None, pair))
      var[i] = Typing(*var[i])
  return vars

_gen_variations_unordered = _gen_variations_unordered_claude

variations = _gen_variations_unordered([ROCK,DRAGON,WATER,FIRE,GRASS,FLYING])
len(variations), variations

(15,
 [[(rock, dragon), (water, fire), (grass, flying)],
  [(rock, dragon), (water, grass), (fire, flying)],
  [(rock, dragon), (water, flying), (fire, grass)],
  [(rock, water), (dragon, fire), (grass, flying)],
  [(rock, water), (dragon, grass), (fire, flying)],
  [(rock, water), (dragon, flying), (fire, grass)],
  [(rock, fire), (dragon, water), (grass, flying)],
  [(rock, fire), (dragon, grass), (water, flying)],
  [(rock, fire), (dragon, flying), (water, grass)],
  [(rock, grass), (dragon, water), (fire, flying)],
  [(rock, grass), (dragon, fire), (water, flying)],
  [(rock, grass), (dragon, flying), (water, fire)],
  [(rock, flying), (dragon, water), (fire, grass)],
  [(rock, flying), (dragon, fire), (water, grass)],
  [(rock, flying), (dragon, grass), (water, fire)]])

# i cheated
unfortunately, i had to use claude to figure out the correct version of `_gen_variations_unordered`... (not for `_gen_variations_ordered` tho)

got close, but still cheated. i hate to make excuses, but this was taking too long and i need to get to my totk content because i have 25 days to get monetized by gathering another 1.2mill views (currently around 1.8). im so sorry 😭

to be fair through, the recursive formulation of the claude algo is the same as the original _ordered algo i made - just considering less combinations... i shoulda approached it that way - optimize the _ordered version...

ok ill stop 🛑

In [36]:
Counter(map(str, _gen_variations_unordered([ROCK,DRAGON,WATER,FIRE,GRASS,FLYING])))

Counter({'[(rock, dragon), (water, fire), (grass, flying)]': 1,
         '[(rock, dragon), (water, grass), (fire, flying)]': 1,
         '[(rock, dragon), (water, flying), (fire, grass)]': 1,
         '[(rock, water), (dragon, fire), (grass, flying)]': 1,
         '[(rock, water), (dragon, grass), (fire, flying)]': 1,
         '[(rock, water), (dragon, flying), (fire, grass)]': 1,
         '[(rock, fire), (dragon, water), (grass, flying)]': 1,
         '[(rock, fire), (dragon, grass), (water, flying)]': 1,
         '[(rock, fire), (dragon, flying), (water, grass)]': 1,
         '[(rock, grass), (dragon, water), (fire, flying)]': 1,
         '[(rock, grass), (dragon, fire), (water, flying)]': 1,
         '[(rock, grass), (dragon, flying), (water, fire)]': 1,
         '[(rock, flying), (dragon, water), (fire, grass)]': 1,
         '[(rock, flying), (dragon, fire), (water, grass)]': 1,
         '[(rock, flying), (dragon, grass), (water, fire)]': 1})

In [37]:
len(Counter(map(str, _gen_variations_ordered([ROCK,DRAGON,WATER,FIRE,GRASS,FLYING]))))

90

In [38]:
def gen_variations(S, ordered=True):
  return (
    _gen_variations_ordered(S)
    if ordered else
    _gen_variations_unordered(S)
  )

# TODO
why does gen_variations not work when theres an odd number of types

In [39]:
gen_variations([FIRE, FLYING, PSYCHIC, ROCK, WATER, ELECTRIC, GROUND], False)

[[(fire, flying), (psychic, rock), (water, electric), (ground)],
 [(fire, flying), (psychic, rock), (water, ground), (electric)],
 [(fire, flying), (psychic, rock), (water), (electric, ground)],
 [(fire, flying), (psychic, water), (rock, electric), (ground)],
 [(fire, flying), (psychic, water), (rock, ground), (electric)],
 [(fire, flying), (psychic, water), (rock), (electric, ground)],
 [(fire, flying), (psychic, electric), (rock, water), (ground)],
 [(fire, flying), (psychic, electric), (rock, ground), (water)],
 [(fire, flying), (psychic, electric), (rock), (water, ground)],
 [(fire, flying), (psychic, ground), (rock, water), (electric)],
 [(fire, flying), (psychic, ground), (rock, electric), (water)],
 [(fire, flying), (psychic, ground), (rock), (water, electric)],
 [(fire, flying), (psychic), (rock, water), (electric, ground)],
 [(fire, flying), (psychic), (rock, electric), (water, ground)],
 [(fire, flying), (psychic), (rock, ground), (water, electric)],
 [(fire, psychic), (flyin

# A Single Function
bringing all the code in one place, and making it run on any input team

(dont forget index in webapp)

In [40]:
from collections import Counter

def team_disadvantages(team):
  weakness_map = []
  for pokemon in team:
    weakness_map.append((
      pokemon,
      disadvantages(*pokemon.typing)
    ))
  return weakness_map

def vs_advantages(my_team, opp_team):
  advantages = []
  for friend in my_team:
    for type in friend:
      for opp in opp_team:
        scalar = type_chart[type][opp.typing[0]]
        if len(opp.typing) > 1:
          scalar *= type_chart[type][opp.typing[1]]
        if scalar > 1:
          advantages.append((type, opp))
  return advantages


def calculate_teams(opponent_team, ordered, team_size):

  opp_disadvantages = team_disadvantages(opponent_team)

  adv_counter = Counter()

  #we can assume each type only appears once per opp
  for _, weaknesses in opp_disadvantages:
    for type in weaknesses:
      adv_counter[type] += 1

  effective_types = list(adv_counter.keys())

  if len(effective_types) < 4:
    raise NotImplementedError("For now, will only work when it produces 4 pokemon")

  max_advs = 0
  choices = min(team_size * 2, len(effective_types))

  strongest_teams = []

  for team in combinations(effective_types, choices):
    for var in gen_variations(team, False):
      advs = len(vs_advantages(var, opponent_team))
      strongest_teams.append((advs, var))

  if False:
    print(len(strongest_teams))

  strongest_teams = sorted(strongest_teams, key=lambda advs_team: advs_team[0])
  max_advs = strongest_teams[-1][0]
  strongest_teams = list(filter(lambda advs_team: advs_team[0] == max_advs, strongest_teams))
  strongest_teams = list(map(lambda advs_team: advs_team[1], strongest_teams))

  return max_advs, strongest_teams

In [41]:
def randomizer(opponent_team, index, team_size=None, ordered=False):
  if team_size is None:
    team_size = len(opponent_team)
  advs, strongest_teams = calculate_teams(opponent_team, ordered, team_size)

  def is_valid_team(team):
    return False not in [index.get(poke, False) for poke in team]

  strongest_teams = list(filter(is_valid_team, strongest_teams))

  random_typings = random.choice(strongest_teams)

  exact_team = []

  for typing in random_typings:
    exact_team.append(random.choice(index[Typing(*typing)]))

  return advs, exact_team

In [42]:
randomizer([
  Typing(DARK, ROCK),
  Typing(WATER, ELECTRIC),
  Typing(FIRE, DARK),
  Typing(WATER, GHOST)
], index)

(14, ['greninja', 'toedscruel', 'annihilape', 'bastiodon'])

> note: im getting 14 advantages because 2+ pokemon in my team are effective against a 1+ pokemon in their team

# Limit to Pokemon Champions

In [43]:
def build_small_index(national, subset):
  index = {
    Typing(*pair): []
    for pair
    in list(combinations(ALL_TYPES, 2)) + [(t,) for t in ALL_TYPES]
  }

  subset = set(subset)

  pokemon_in_subset = filter(lambda poke: poke['name'] in subset, national)

  for pokemon in pokemon_in_subset:
    typing = Typing(*list(map(type_name_to_type.__getitem__, pokemon['typing'])))
    index[typing].append(pokemon['name'])
  
  typings = list(index.keys())
  for typing in typings:
    if not index[typing]:
      del index[typing]
  
  return index

#just realized pokedex is derived from index

In [44]:
import json

with open('../app/data/champions.json') as f:
  champions_roster = json.loads(f.read())

champions_index = build_small_index(all_pokemon, champions_roster)
champions_index

{(normal, fire): ['pyroar-male'],
 (normal, electric): ['heliolisk'],
 (normal, ground): ['diggersby'],
 (normal, flying): ['pidgeot', 'staraptor', 'toucannon'],
 (normal, psychic): ['oranguru', 'wyrdeer', 'farigiraf'],
 (normal, ghost): ['zoroark-hisui'],
 (normal, dragon): ['drampa'],
 (fire, electric): ['rotom-heat'],
 (fire, grass): ['scovillain'],
 (fire, fighting): ['blaziken',
  'infernape',
  'emboar',
  'tauros-paldea-blaze-breed'],
 (fire, poison): ['salazzle'],
 (fire, ground): ['camerupt'],
 (fire, flying): ['charizard', 'talonflame'],
 (fire, psychic): ['delphox', 'armarouge'],
 (fire, bug): ['volcarona'],
 (fire, rock): ['arcanine-hisui'],
 (fire, ghost): ['chandelure', 'skeledirge', 'ceruledge', 'typhlosion-hisui'],
 (fire, dark): ['houndoom', 'incineroar'],
 (water, electric): ['rotom-wash'],
 (water, fighting): ['quaquaval', 'tauros-paldea-aqua-breed'],
 (water, poison): ['qwilfish', 'toxapex'],
 (water, ground): ['swampert'],
 (water, flying): ['gyarados', 'pelipper']

In [45]:
# INPUT_TEAM = []

randomizer(INPUT_TEAM, champions_index)

(10, ['garchomp', 'pelipper', 'dedenne', 'abomasnow'])

In [46]:
randomizer([
    Typing(POISON, GRASS),
    Typing(FLYING, FIRE),
    Typing(WATER),
    Typing(BUG,POISON),
  ], champions_index, ordered=True)

(13, ['arcanine-hisui', 'abomasnow', 'emolga', 'starmie'])

In [47]:
#should produce similar output to above
randomizer([
  Typing(POISON, GRASS),
  Typing(FLYING, FIRE),
  Typing(WATER),
  Typing(BUG,POISON),
], champions_index, ordered=False)

(13, ['scovillain', 'mr-rime', 'aerodactyl', 'rotom-wash'])

In [48]:
randomizer([
  Typing(POISON, FIGHTING), #sneasler
  Typing(ROCK, FLYING),     #aerodactyl
  Typing(GROUND, DRAGON),   #garchomp
  Typing(WATER, ELECTRIC),  #rotom-wash
], champions_index, ordered=True)

(10, ['torterra', 'aerodactyl', 'slowking', 'ninetales-alola'])

In [49]:
#this implementation is wayyy faster
randomizer([
  Typing(POISON, FIGHTING), #sneasler
  Typing(ROCK, FLYING),     #aerodactyl
  Typing(GROUND, DRAGON),   #garchomp
  Typing(WATER, ELECTRIC),  #rotom-wash
], champions_index, ordered=False)

(10, ['steelix', 'raichu-alola', 'ninetales-alola', 'appletun'])

In [50]:
#this implementation is wayyy faster
randomizer([
  Typing(POISON, BUG),
  Typing(FIRE, FLYING),
  Typing(POISON),
  Typing(ELECTRIC),
], champions_index, ordered=False)

(10, ['camerupt', 'aerodactyl', 'slowking', 'eelektross'])

# Team Size
parameterized the output team size

In [51]:
#this implementation is wayyy faster
if False:
  randomizer([
    Typing(POISON, FIGHTING), #sneasler
    Typing(ROCK, FLYING),     #aerodactyl
    Typing(GROUND, DRAGON),   #garchomp
    Typing(WATER, ELECTRIC),  #rotom-wash
    Typing(STEEL, GROUND),    #excadrill
    Typing(ROCK, DARK),       #ttarc
  ], champions_index, ordered=False)

# New Goal

Find the team with maximum coverage across all possible opposing teams...

👉 to ensure we're considering all teams that beat all 76 opposing types, here's the algo:
1. go through each combo of 4
2. use another version of vs_advantages that returns amount of opposing types that our team beats
3. just filter out the teams that beat all 76 types


In [52]:
def wins(my_team, opp_team):
  # assuming no duplicate types in the opp team
  
  advs = set(opp_team)

  for friend in my_team:
    for type in friend:
      for opp in opp_team:
        scalar = type_chart[type][opp.typing[0]]
        if len(opp.typing) > 1:
          scalar *= type_chart[type][opp.typing[1]]
        if scalar > 1 and opp in advs:
          advs.remove(opp)
  return set(opp_team) - advs

w = wins(
  [Typing(ROCK)],
  list(champions_index.keys())
)
len(w), w

(29,
 {(bug),
  (bug, rock),
  (electric, flying),
  (electric, ice),
  (fire),
  (fire, bug),
  (fire, dark),
  (fire, electric),
  (fire, flying),
  (fire, ghost),
  (fire, grass),
  (fire, poison),
  (fire, psychic),
  (fire, rock),
  (flying, bug),
  (flying, dragon),
  (flying, rock),
  (grass, ice),
  (ice),
  (ice, dark),
  (ice, fairy),
  (ice, ghost),
  (ice, psychic),
  (ice, rock),
  (normal, fire),
  (normal, flying),
  (poison, bug),
  (water, bug),
  (water, flying)})

In [ ]:
import os

from collections import defaultdict

teams_that_cover_all_types = defaultdict(lambda: [])

all_possible_champions_typings = list(champions_index.keys())
max_advs = 0
best_teams = []

filename_best_team = './tmp/best-teams.json'

def encode_best_teams(best_teams):
  with open(filename_best_team, 'w') as f:
    f.write(str([
      [list(map(int,poke.typing)) for poke in team]
      for team in best_teams
    ]))

def decode_best_teams():
  best_teams = []
  with open(filename_best_team) as f:
    for team in json.loads(f.read()):
      best_teams.append([
        Typing(*poke)
        for poke
        in team
      ])
  return best_teams


if not os.path.exists(filename_best_team):

  for team in combinations(ALL_TYPES, 8):
    for variant in gen_variations(team):
      valid_champions_types = False not in [poke in champions_index for poke in variant]
      if valid_champions_types:
        advs = len(wins(variant, all_possible_champions_typings))
        if advs > max_advs:
          max_advs = advs
          best_teams.clear()
        if advs == max_advs:
          best_teams.append(variant)

  encode_best_teams(best_teams)

else:
  best_teams = decode_best_teams()


len(best_teams), best_teams[:min(5, len(best_teams))],'...'

(1776,
 [[(rock, dark), (ground, water), (steel, fairy), (fire, grass)],
  [(water, dark), (ground, rock), (steel, fairy), (fire, grass)],
  [(water, rock), (ground, dark), (steel, fairy), (fire, grass)],
  [(ground, dark), (water, rock), (steel, fairy), (fire, grass)],
  [(ground, rock), (water, dark), (steel, fairy), (fire, grass)]],
 '...')

In [54]:
len(all_possible_champions_typings)

111

In [70]:
max_advs = len(next(iter(best_teams)))
for team in best_teams:
  max_advs = max(
    max_advs,
    len(wins(team, all_possible_champions_typings))
  )

max_advs

110

# 92
the max advantages that any team of 4 covers out of all potential opponents

choose(ALL_TYPES,8) against all 93 typings

TODO: what is the type that the team DOESNT cover? probably dark ghost. or normal ghost.

# TODO
memoize the count_wins func

In [55]:
f'there are {len(best_teams)} best teams'

'there are 1776 best teams'

In [56]:
team_and_safe = {}

for team in best_teams:
  types_we_beat = wins(team, all_possible_champions_typings)
  types_we_dont_beat = set(all_possible_champions_typings) - types_we_beat

  team_and_safe[str(types_we_dont_beat)] = str(team)


team_and_safe

{'{(normal)}': '[(ground, fire), (grass, ice), (rock, dark), (steel, fairy)]',
 '{(ground, flying)}': '[(ground, fire), (grass, fighting), (rock, dark), (steel, fairy)]'}

In [57]:
best_teams_file = './tmp/best_teams_and_what_they_miss.json'
if not os.path.exists(best_teams_file):
  with open(best_teams_file, 'w+') as f:
    f.write(json.dumps(team_and_safe))
else:
  print('skipped')

skipped


# conclusion
Is there a team in pokemon champions that covers against every possible type combo in the game?

> No.

but there ARE 3888 type combos (even more if youre talking specific teams) (❗️) that cover almost every type in the game, "almost" meaning all but one.

now keep in mind, any of the pokemon in this list can likely learn a move that will cover that remaining type, this is more just about generating a team that will give your opponent a hard time when theyre picking a team in the team selection screen.

# scrap
the following is scrap

In [58]:
i = 0

for team in combinations(champions_roster, 6):
  # no need to teamify- thats for typings
  if i >= 5e5:
    break
  i += 1



5e7 -> 4.6s

In [59]:
# wins_pokemon: same as wins() but takes Pokemon objects
# all_pokemon structure: [{"name": "venusaur", "typing": ["grass", "poison"]}, ...]

class Pokemon:
  def __init__(self, name, typing):
    self.name = name
    self.typing = typing  # list of type name strings e.g. ["fire", "flying"]
  def __repr__(self):
    return f"Pokemon({self.name}, {self.typing})"
  def __hash__(self):
    return hash(self.name)
  def __eq__(self, other):
    return isinstance(other, Pokemon) and self.name == other.name


def wins_pokemon(my_team, opp_team):
  """
  my_team:  list of Pokemon objects
  opp_team: list of Pokemon objects to check coverage against
  Returns a list of (our_pokemon, our_pokemon_type, opp_pokemon) tuples
  for every super-effective hit. A single matchup can appear twice if
  both of our_pokemon's types beat the opp.
  """
  results = []

  for friend in my_team:
    for type_name in friend.typing:
      attack_type = type_name_to_type[type_name]
      for opp in opp_team:
        def_type0 = type_name_to_type[opp.typing[0]]
        scalar = type_chart[attack_type][def_type0]
        if len(opp.typing) > 1:
          scalar *= type_chart[attack_type][type_name_to_type[opp.typing[1]]]
        if scalar > 1:
          results.append((friend, type_name, opp))

  return results


# Build champions pool as Pokemon objects from all_pokemon
champions_set = set(champions_roster)
champions_pool = [
  Pokemon(p["name"], p["typing"])
  for p in all_pokemon
  if p["name"] in champions_set
]

print(f"Champions pool size: {len(champions_pool)}")
print(champions_pool[:3], "...")


# Updated loop: iterate over Pokemon objects instead of name strings
i = 0
for team in combinations(champions_pool, 6):
  # team is now a tuple of Pokemon objects
  if i >= 5e4:
    break
  wins_pokemon(list(team), champions_pool)
  i += 1


Champions pool size: 236
[Pokemon(venusaur, ['grass', 'poison']), Pokemon(charizard, ['fire', 'flying']), Pokemon(blastoise, ['water'])] ...
